# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides an end-to-end workflow for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is described by a Croissant schema JSON-LD file:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset and its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not treat as dict or list

# Print summary
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s.

We enumerate record sets and for each, show field `@id`s and field labels.

In [ ]:
# List all record sets and their fields, with `@id` references
if hasattr(dataset, "record_sets"):
    record_sets = dataset.record_sets
else:
    # For compatibility with older mlcroissant versions
    record_sets = dataset._record_sets

print("Record sets and their field @id(s):\n---------------------------")
for rs in record_sets:
    print(f"- RecordSet label: {getattr(rs, 'name', '[no-name]')} (@id: {rs.id})")
    print("  Fields:")
    for f in rs.fields:
        fname = getattr(f, 'name', '[no-name]')
        print(f"    - {fname} (@id: {f.id})")
    print()

# For quick reference, collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis, referencing by their `@id` values.

In [ ]:
# Extract all dataframes indexed by record set @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# List columns and show sample for the first record set
if record_set_ids:
    main_record_set = record_set_ids[0]
    print(f"Columns for record set @id '{main_record_set}':")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalization, and grouping. All fields are referenced by their `@id`.

In [ ]:
# Example: filter by a numeric field and normalize it
# Find a numeric field from the main record set:
main_fields = [f for f in record_sets[0].fields]
numeric_field = None
for f in main_fields:
    # Try to get dataType if available
    try:
        dtype = str(getattr(f, 'data_type', '')).lower()
        if 'float' in dtype or 'integer' in dtype or 'number' in dtype:
            numeric_field = f.id
            break
    except Exception:
        pass

df = dataframes[main_record_set]

if numeric_field and numeric_field in df.columns:
    # Remove obvious missing values and filter
    threshold = 0
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
    print(f"Filtered records where `{numeric_field}` > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field], errors='coerce') - 
        pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
    print(f"\nNormalized `{numeric_field}` for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Choose a group field (try for the second field if possible and not numeric)
    group_field = None
    for f in main_fields:
        if f.id != numeric_field and f.id in df.columns:
            sample_val = df[f.id].dropna().astype(str).values
            if len(sample_val) > 0 and not sample_val[0].replace('.', '', 1).isdigit():
                group_field = f.id
                break

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped by `{group_field}`:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found in this record set for EDA.")

## 5. Visualization
Visualize the distribution of a selected numeric field, or show a bar chart of group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field and group_field must come from previous cell
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

if 'grouped_df' in locals() and group_field:
    grouped_df_sorted = grouped_df.sort_values(numeric_field, ascending=False)
    plt.figure(figsize=(8,5))
    sns.barplot(x=grouped_df_sorted.index, y=grouped_df_sorted[numeric_field], palette='Blues')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and analyze the FAIR² dataset using the `mlcroissant` library, referencing all dataset structures by their `@id`. We explored available record sets, fields, and extracted records, applying filtering, normalization, and basic visualization. 

For more advanced analysis, consult the Croissant schema documentation and experiment with additional fields or derived features in the dataset.